# 19 — Supervisor with Real Tools

Workers that actually **search the web**, **run code**, and **fetch pages**. Three ways to give workers tools: `with_builtins` (all tools), specific tools, or `Agent.with_builtins`.

**What you'll learn:**
1. Supervisor.with_builtins — workers get all 30+ tools automatically
2. Manual tool assignment — pick specific tools per worker
3. Agent.with_builtins per worker — full control
4. Streaming output with real tool calls visible

In [ ]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent, WebSearchTool, CodeExecutionTool, OpenURLTool
from shipit_agent.deep import Supervisor, Worker
from examples.run_multi_tool_agent import build_llm_from_env

llm = build_llm_from_env("bedrock")


def print_event(event):
    """Pretty-print with output content."""
    ICONS = {
        "run_started": "🚀",
        "run_completed": "✅",
        "planning_started": "📋",
        "step_started": "▶️",
        "tool_completed": "📦",
        "tool_called": "🔧",
        "tool_failed": "❌",
    }
    icon = ICONS.get(event.type, "•")
    worker = event.payload.get("worker", "")
    prefix = f"[{worker}] " if worker else ""
    print(f"\n{icon} {prefix}{event.message}")
    output = event.payload.get("output", "")
    if output and event.type in ("tool_completed", "run_completed"):
        print(f"{'─' * 60}")
        print(output[:600])
        if len(output) > 600:
            print(f"... ({len(output)} chars)")
        print(f"{'─' * 60}")

## 1. Supervisor.with_builtins — Easiest Way

One line gives ALL workers web search, code execution, URL fetching, memory, planner, and 25+ more tools.

In [ ]:
# Method 1: with_builtins — ALL workers get ALL tools automatically
supervisor = Supervisor.with_builtins(
    llm=llm,
    worker_configs=[
        {
            "name": "researcher",
            "prompt": "You are a research analyst. Use web_search to find real data. Be concise.",
            "capabilities": ["web search", "research", "data analysis"],
        },
        {
            "name": "writer",
            "prompt": "You are a technical writer. Write clear, engaging content from research data.",
            "capabilities": ["writing", "reports", "summaries"],
        },
    ],
    max_delegations=4,
)

print("=== Supervisor.with_builtins — Workers Search the Web ===\n")
for event in supervisor.stream(
    "Research Python's popularity in 2025 and write a 2-paragraph summary with real data"
):
    print_event(event)

## 2. Manual Tool Assignment — Pick Specific Tools

Give the researcher web search + URL fetching, but give the writer no tools (just writes from context).

In [ ]:
# Method 2: Manual — researcher gets web tools, writer gets none
researcher = Worker(
    name="researcher",
    agent=Agent(
        llm=llm,
        prompt="You are a research analyst. Use web_search to find real data and open_url to read pages.",
        tools=[WebSearchTool(), OpenURLTool()],  # specific tools
    ),
    capabilities=["web search", "research"],
)

coder = Worker(
    name="coder",
    agent=Agent(
        llm=llm,
        prompt="You are a Python developer. Use code_execution to run code and verify results.",
        tools=[CodeExecutionTool()],  # only code execution
    ),
    capabilities=["coding", "python", "data analysis"],
)

writer = Worker(
    name="writer",
    agent=Agent(
        llm=llm, prompt="You write clear reports. No tools needed — just write."
    ),
    # NO tools — writer just uses LLM
)

supervisor = Supervisor(llm=llm, workers=[researcher, coder, writer], max_delegations=5)

print("=== Manual Tools — Researcher Searches, Coder Runs Code, Writer Writes ===\n")
for event in supervisor.stream(
    "Find the current Python version, verify it with code, then write a summary"
):
    print_event(event)

## 3. Agent.with_builtins Per Worker

Each worker gets a full-power agent with all tools. Most flexible for production use.

In [ ]:
# Method 3: Agent.with_builtins per worker — full power, full control
analyst = Worker(
    name="market-analyst",
    agent=Agent.with_builtins(
        llm=llm,
        prompt="You are a market analyst. Search the web for real market data, trends, and statistics.",
    ),
    capabilities=["market research", "data", "web search"],
)

content_creator = Worker(
    name="content-creator",
    agent=Agent.with_builtins(
        llm=llm,
        prompt="You create viral social media content. Use web search for trending topics.",
    ),
    capabilities=["content", "social media", "trends"],
)

supervisor = Supervisor(llm=llm, workers=[analyst, content_creator], max_delegations=4)

print("=== Full-Power Workers — Agent.with_builtins ===\n")
for event in supervisor.stream(
    "Research AI agent frameworks trending on GitHub, then create 3 tweet-sized posts"
):
    print_event(event)